# 02b GlyBERTa Tokenizer Generation

## Purpose

This notebook trains the GlyBERTa-style tokenizer for the rebuilt workflow.

## Input

- `MyDrive/ProjectRoot/data/splits/train.txt`

## Outputs

- `MyDrive/ProjectRoot/tokenizers/glyberta/<setting_label>/vocab.json`
- Hugging Face tokenizer files saved in the same folder
- `tokenizer_config_summary.json`
- `inspection_preview.csv`

## Notes to myself

This tokenizer follows the GlyBERTa idea of splitting glycans into glyco-letters. In this rebuilt workflow, that idea is adapted to the compact IUPAC strings in the project dataset, so inline linkage text such as `b1-4` and branch markers such as `(`, `)`, `[` and `]` become explicit tokens before the WordLevel vocabulary is learned from the training split.


## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenizer artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read data files and save outputs.
drive.mount('/content/drive')

# Clone the public GitHub repository into the Colab runtime.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}
!git pull origin main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


## Path and setting setup

This tokenizer does not have a BPE merge schedule, so I am using a fixed label that reflects the workflow rather than a vocab-size hyperparameter. `v1_train_only` means the adapted glyco-letter vocabulary was learned from the training split only.


In [ ]:
# ==============================================================================
# 1. DEFINE THE TRAINING PATHS AND TOKENIZER SETTINGS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot'
TRAIN_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'splits', 'train.txt')

SETTING_LABEL = 'v1_train_only'
TOKENIZER_OUT_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', 'glyberta', SETTING_LABEL)
os.makedirs(TOKENIZER_OUT_DIR, exist_ok=True)

print('Training data path:')
print(TRAIN_DATA_PATH)
print('\nTokenizer output directory:')
print(TOKENIZER_OUT_DIR)
print('\nSetting label:')
print(SETTING_LABEL)

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f'Training split not found: {TRAIN_DATA_PATH}')


## Train the tokenizer

This is the actual GlyBERTa-style training step. The logic lives in `src/tokenizer_utils.py` so the tokenizer definition can be reused elsewhere without copying notebook code. The helper uses an adapted split rule for this project's compact glycan strings rather than the exact string format used in the original GlyBERTa script.


In [ ]:
# ==============================================================================
# 2. TRAIN AND SAVE THE GLYBERTA-STYLE TOKENIZER
# ==============================================================================
import importlib
import json

if 'src.tokenizer_utils' in sys.modules:
    importlib.reload(sys.modules['src.tokenizer_utils'])

from src.tokenizer_utils import GLYBERTA_COMPACT_GLYCOLETTER_PATTERN, train_glyberta_wordlevel

print('Training GlyBERTa-style WordLevel tokenizer...')

backend_tokenizer, hf_tokenizer = train_glyberta_wordlevel(TRAIN_DATA_PATH)
backend_tokenizer.model.save(TOKENIZER_OUT_DIR)
hf_tokenizer.save_pretrained(TOKENIZER_OUT_DIR)

tokenizer_summary = {
    'tokenizer_family': 'glyberta',
    'setting_label': SETTING_LABEL,
    'train_data_path': TRAIN_DATA_PATH,
    'tokenizer_output_dir': TOKENIZER_OUT_DIR,
    'pretokenizer_pattern': GLYBERTA_COMPACT_GLYCOLETTER_PATTERN,
    'vocab_size': len(hf_tokenizer),
    'saved_files': [
        'tokenizer.json',
        'tokenizer_config.json',
        'vocab.json',
        'tokenizer_config_summary.json',
        'inspection_preview.csv',
    ],
}

summary_json_path = os.path.join(TOKENIZER_OUT_DIR, 'tokenizer_config_summary.json')
with open(summary_json_path, 'w', encoding='utf-8') as file:
    json.dump(tokenizer_summary, file, indent=2)

print('Tokenizer training complete.')
print(f'Tokenizer saved to: {TOKENIZER_OUT_DIR}')


## Quick sanity check

I do not want to do deep analysis here. I just want to confirm the tokenizer loads, has the expected vocabulary size, and produces reasonable glyco-letter splits on a few example glycans.


In [ ]:
# ==============================================================================
# 3. LOAD THE SAVED TOKENIZER AND INSPECT SAMPLE OUTPUT
# ==============================================================================
import pandas as pd
from transformers import PreTrainedTokenizerFast

with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as file:
    train_sequences = [line.strip() for line in file if line.strip()]

loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_OUT_DIR)

sample_sequences = train_sequences[:3]
inspection_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = loaded_tokenizer.encode(sequence, add_special_tokens=False)
    tokens = loaded_tokenizer.convert_ids_to_tokens(token_ids)

    inspection_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:30]),
        }
    )

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')


## Save a small inspection table

This gives me a lightweight record of the first sanity check without reopening the notebook later.


In [ ]:
# ==============================================================================
# 4. SAVE THE INSPECTION OUTPUT
# ==============================================================================
inspection_path = os.path.join(TOKENIZER_OUT_DIR, 'inspection_preview.csv')
inspection_df.to_csv(inspection_path, index=False)

print(f'Inspection preview saved to: {inspection_path}')
